In [0]:
dbutils.widgets.dropdown("data_source", "products", 
    ["products", "categories", "inventory", "clickstream", "orders", "order_items", "customers"], 
    "Data Source")

dbutils.widgets.dropdown("catalog", "dev", ["dev", "prod"])

data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")

print(f"Selected source: {data_source} and catalog: {catalog}")

In [0]:
from pyspark.sql import functions as F

In [0]:
rules = {
    "orders": {
        "valid_order_id": "after.order_id IS NOT NULL",
        "valid_customer_ref": "after.customer_id IS NOT NULL",
        "valid_total_amount": "(after.total_amount IS NULL OR after.total_amount >= 0)",
    },
    "products": {
        "valid_product_id": "product_id IS NOT NULL",
        "valid_sku": "sku IS NOT NULL",
        "valid_retail_price": "(retail_price IS NULL OR retail_price > 0)",
    },
    "order_items": {
        "valid_order_item_id": "after.order_item_id IS NOT NULL",
        "valid_order_ref": "after.order_id IS NOT NULL",
        "valid_product_ref": "after.product_id IS NOT NULL",
        "valid_quantity": "(after.quantity IS NULL OR after.quantity > 0)",
    },
    "customers": {
        "valid_customer_id": "after.customer_id IS NOT NULL",
        "valid_email": "after.email IS NOT NULL",
        "valid_loyalty_tier": "(after.loyalty_tier IS NULL OR after.loyalty_tier IN ('bronze','silver','gold','platinum'))",
    },
    "categories": {
        "valid_category_id": "category_id IS NOT NULL",
        "valid_category_name": "category_name IS NOT NULL",
    },
    "clickstream": {
        "valid_event_id": "event_id IS NOT NULL",
        "valid_event_type": "event_type IS NOT NULL",
        "valid_event_timestamp": "event_timestamp IS NOT NULL",
    },
    "inventory": {
        "valid_snapshot_id": "snapshot_id IS NOT NULL",
        "valid_product_ref": "product_id IS NOT NULL",
        "valid_quantity_on_hand": "(quantity_on_hand IS NULL OR quantity_on_hand >= 0)",
    },
}

In [0]:
SOURCE_TABLE_MAP = {
    "products": "bronze_products",
    "categories": "bronze_categories",
    "inventory": "bronze_inventory",
    "clickstream": "bronze_clickstream",
    "orders": "bronze_orders_cdc",
    "order_items": "bronze_order_items_cdc",
    "customers": "bronze_customers_cdc",
}

source_table = f"{catalog}.os_stepright.{SOURCE_TABLE_MAP[data_source]}"

print(source_table)

In [0]:
def rules_expr(data_source):

    return ([' AND '.join(i) for i in [rules[data_source].values()]][0])



In [0]:
def read_source_table(source_table, data_source):

    source_df = (
        spark.readStream.table(source_table)
        .withColumn("is_valid", F.expr(rules_expr(data_source)))
    )

    return source_df

In [0]:
def write_to_targets(source_df, catalog, data_source):

    valid_df = source_df.filter(F.col("is_valid") == True)
    (valid_df.writeStream
        .format("delta")
        .option("checkpointLocation", f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{data_source}_valid")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(f"{catalog}.os_stepright.bronze_{data_source}_valid")
    )

    quarantined_df = source_df.filter(F.col("is_valid") == False)
    (quarantined_df.writeStream
        .format("delta")
        .option("checkpointLocation", f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{data_source}_quarantined")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(f"{catalog}.os_stepright.bronze_{data_source}_quarantined")
    )

In [0]:
source_df = read_source_table(source_table, data_source)
write_to_targets(source_df, catalog, data_source)

###Validation after run

In [0]:
invalid_data_check_query = f"select * from {catalog}.os_stepright.bronze_{data_source}_quarantined"
print(invalid_data_check_query)


In [0]:
spark.sql(invalid_data_check_query).display()

In [0]:
valid_data_check_query = f"select * from {catalog}.os_stepright.bronze_{data_source}_valid"
print(valid_data_check_query)

In [0]:
spark.sql(invalid_data_check_query).display()

In [0]:
# catalog = "dev"  # change if needed

# # --- Bronze tables ---
# bronze_file_sources = ["products", "categories", "inventory", "clickstream"]
# bronze_cdc_sources = ["orders", "order_items", "customers"]

# # --- Pre-Silver DQ sources (all 7) ---
# dq_sources = ["products", "categories", "inventory", "clickstream", "orders", "order_items", "customers"]

# def drop_and_clear(target_table, checkpoint_loc):
#     spark.sql(f"DROP TABLE IF EXISTS {target_table}")
#     print(f"Dropped table: {target_table}")
#     try:
#         dbutils.fs.rm(checkpoint_loc, recurse=True)
#         print(f"Cleared checkpoint: {checkpoint_loc}")
#     except Exception as e:
#         print(f"No checkpoint found at {checkpoint_loc} (skipping)")

# print("=== Clearing Bronze (file-based) ===")
# for source in bronze_file_sources:
#     target_table = f"{catalog}.os_stepright.bronze_{source}"
#     checkpoint_loc = f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{source}"
#     drop_and_clear(target_table, checkpoint_loc)

# print("\n=== Clearing Bronze (CDC) ===")
# for source in bronze_cdc_sources:
#     target_table = f"{catalog}.os_stepright.bronze_{source}_cdc"
#     checkpoint_loc = f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{source}_cdc"
#     drop_and_clear(target_table, checkpoint_loc)

# print("\n=== Clearing Pre-Silver DQ (valid + quarantined) ===")
# for source in dq_sources:
#     valid_table = f"{catalog}.os_stepright.bronze_{source}_valid"
#     quarantined_table = f"{catalog}.os_stepright.bronze_{source}_quarantined"
#     valid_checkpoint = f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{source}_valid"
#     quarantined_checkpoint = f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{source}_quarantined"

#     drop_and_clear(valid_table, valid_checkpoint)
#     drop_and_clear(quarantined_table, quarantined_checkpoint)

# print("\nAll Bronze and Pre-Silver DQ tables and checkpoints cleared.")